In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
! pip install datasets

In [2]:
from datasets import load_dataset

dataset = load_dataset("Skylion007/openwebtext", split="train", cache_dir="/content/drive/MyDrive/openwebtext_cache")

print("Dataset loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/80 [00:00<?, ?it/s]

Dataset loaded!


In [3]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.10.0+cu128


In [4]:
import torch 
from torch import nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 32
block_size = 64 
max_iters = 500
learning_rate = 1e-3
eval_iters = 50 
n_embd=128
n_head=4
n_layer=3
dropout = 0.2 
print(f"Using device: {device}")


Using device: cuda


In [5]:
import argparse
parser = argparse.ArgumentParser(description='Train a GPT-like model on the OpenWebText dataset.')
parser.add_argument('--train_data', type=str, default='/content/output_train.txt', help='Path to the training data file')
parser.add_argument('--val_data', type=str, default='/content/output_val.txt', help='Path to the validation data file')
parser.add_argument('--batch_size', type=int, default=8, help='Batch size for training')
args = parser.parse_args()
print(f'batch_size: {args.batch_size}')

usage: colab_kernel_launcher.py [-h] [--train_data TRAIN_DATA]
                                [--val_data VAL_DATA]
                                [--batch_size BATCH_SIZE]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-7d84b1a9-37dd-406f-bcda-1826d1206505.json


SystemExit: 2

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os
import random
from tqdm.notebook import tqdm

train_ratio = 0.9
sample_rate = 0.01
seed = 42
text_column = "text"

dataset_name = "Skylion007/openwebtext"
cache_dir = "/content/drive/MyDrive/openwebtext_cache"
output_dir = cache_dir

output_file_train = os.path.join(output_dir, "output_train.txt")
output_file_val = os.path.join(output_dir, "output_val.txt")
vocab_file = os.path.join(output_dir, "vocab.txt")

In [9]:
def build_samples_from_dataset(ds, train_ratio=0.9, sample_rate=0.01, seed=42, text_column="text"):
    ds = ds.shuffle(seed=seed)
    total = len(ds)
    if total == 0:
        return [], [], set()

    split_index = int(total * train_ratio)
    split_index = max(1, min(split_index, total - 1)) if total > 1 else total

    train_ds = ds.select(range(split_index))
    val_ds = ds.select(range(split_index, total)) if split_index < total else ds.select([])

    train_n = max(1, int(len(train_ds) * sample_rate)) if len(train_ds) else 0
    val_n = max(1, int(len(val_ds) * sample_rate)) if len(val_ds) else 0

    train_ids = random.sample(range(len(train_ds)), train_n) if train_n else []
    val_ids = random.sample(range(len(val_ds)), val_n) if val_n else []

    train_texts = [str(train_ds[i].get(text_column, "")) for i in train_ids]
    val_texts = [str(val_ds[i].get(text_column, "")) for i in val_ids]

    vocab = set()
    for t in train_texts:
        vocab.update(t)
    for t in val_texts:
        vocab.update(t)

    return train_texts, val_texts, vocab


def write_lines(path, lines):
    parent = os.path.dirname(path) or "."
    os.makedirs(parent, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for line in tqdm(lines, desc=f"Writing {path}"):
            f.write(line + "\n")


def write_vocab(path, vocab):
    parent = os.path.dirname(path) or "."
    os.makedirs(parent, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for ch in sorted(vocab):
            f.write(ch)

In [10]:
from datasets import load_dataset

if "dataset" not in globals() or dataset is None:
    print("`dataset` variable not found. Auto-loading from your fixed cache path...")

    if not os.path.isdir("/content/drive/MyDrive"):
        print("Warning: /content/drive/MyDrive not found. Run drive.mount('/content/drive') first.")

    os.makedirs(cache_dir, exist_ok=True)
    print(f"Loading dataset '{dataset_name}' with cache_dir={cache_dir}")
    dataset = load_dataset(dataset_name, split="train", cache_dir=cache_dir)

print("Dataset ready. Building samples...")

os.makedirs(output_dir, exist_ok=True)

train_texts, val_texts, vocab = build_samples_from_dataset(
    dataset,
    train_ratio=train_ratio,
    sample_rate=sample_rate,
    seed=seed,
    text_column=text_column,
)

write_lines(output_file_train, train_texts)
write_lines(output_file_val, val_texts)
write_vocab(vocab_file, vocab)

print(f"Train samples: {len(train_texts)}")
print(f"Val samples: {len(val_texts)}")
print(f"Vocab size: {len(vocab)}")
print(f"Saved files: {output_file_train}, {output_file_val}, {vocab_file}")

Dataset ready. Building samples...


Writing /content/drive/MyDrive/openwebtext_cache/output_train.txt:   0%|          | 0/72123 [00:00<?, ?it/s]

Writing /content/drive/MyDrive/openwebtext_cache/output_val.txt:   0%|          | 0/8013 [00:00<?, ?it/s]

Train samples: 72123
Val samples: 8013
Vocab size: 5311
Saved files: /content/drive/MyDrive/openwebtext_cache/output_train.txt, /content/drive/MyDrive/openwebtext_cache/output_val.txt, /content/drive/MyDrive/openwebtext_cache/vocab.txt


In [11]:
import os
print(os.path.abspath(vocab_file))
print(os.path.exists(vocab_file))


/content/drive/MyDrive/openwebtext_cache/vocab.txt
True


In [12]:
import os

vocab_path = vocab_file if "vocab_file" in globals() else "/content/drive/MyDrive/openwebtext_cache/vocab.txt"
if not os.path.exists(vocab_path):
    raise FileNotFoundError(f"vocab file not found: {vocab_path}. Run the preprocessing/export cell first.")

with open(vocab_path, "r", encoding="utf-8") as f:
    vocab_text = f.read()

chars = sorted(set(vocab_text))
if "\n" not in chars:
    chars.append("\n")
    chars = sorted(chars)

string_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_string = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: "".join([int_to_string[i] for i in l])

vocab_size = len(chars)
print(f"Loaded vocab from: {vocab_path}")
print(f"vocab_size: {vocab_size}")
print("encode('hello world'):", encode("hello world"))

Loaded vocab from: /content/drive/MyDrive/openwebtext_cache/vocab.txt
vocab_size: 5311
encode('hello world'): [73, 70, 77, 77, 80, 1, 88, 80, 83, 77, 69]


In [13]:
import mmap 
import random 

In [14]:
import os


def ensure_encoder_loaded():
    global string_to_int, int_to_string, encode, decode, vocab_size

    required_names = ("string_to_int", "int_to_string", "encode", "decode", "vocab_size")
    if all(name in globals() for name in required_names):
        return

    vocab_path = vocab_file if "vocab_file" in globals() else "/content/drive/MyDrive/openwebtext_cache/vocab.txt"
    if not os.path.exists(vocab_path):
        raise FileNotFoundError(
            f"Tokenizer is not initialized and vocab file was not found: {vocab_path}. "
            "Run the preprocessing/export cell first."
        )

    with open(vocab_path, "r", encoding="utf-8") as f:
        vocab_text = f.read()

    chars = sorted(set(vocab_text))
    if "\n" not in chars:
        chars.append("\n")
        chars = sorted(chars)

    string_to_int = {ch: i for i, ch in enumerate(chars)}
    int_to_string = {i: ch for i, ch in enumerate(chars)}
    encode = lambda s: [string_to_int[c] for c in s]
    decode = lambda tokens: "".join(int_to_string[i] for i in tokens)
    vocab_size = len(chars)


def load_encoded_split(path):
    ensure_encoder_loaded()

    if not os.path.exists(path):
        raise FileNotFoundError(f"Data file not found: {path}. Run the preprocessing/export cell first.")

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    if len(text) <= block_size:
        raise ValueError(
            f"Data in {path} is too small for block_size={block_size}. "
            "Increase sample_rate or reduce block_size."
        )

    try:
        ids = torch.tensor(encode(text), dtype=torch.long)
    except KeyError as e:
        raise KeyError(
            f"Character {repr(e.args[0])} is not in vocab. Regenerate vocab/output files from the same data."
        )
    return ids


train_path = output_file_train if "output_file_train" in globals() else "/content/output_train.txt"
val_path = output_file_val if "output_file_val" in globals() else "/content/output_val.txt"

train_data = load_encoded_split(train_path)
val_data = load_encoded_split(val_path)

print(f"Loaded train tokens: {len(train_data)}, val tokens: {len(val_data)}")


def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

Loaded train tokens: 351844917, val tokens: 39270408


In [16]:


class SingleHeadCausalSelfAttention(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.q_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.k_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.v_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        q = self.q_proj(x).unsqueeze(1)
        k = self.k_proj(x).unsqueeze(1)
        v = self.v_proj(x).unsqueeze(1)

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=dropout if self.training else 0.0,
            is_causal=True,
        ).squeeze(1)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.k_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.v_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.size()

        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=dropout if self.training else 0.0,
            is_causal=True,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, use_multi_head=True):
        super().__init__()
        if use_multi_head and n_head > 1:
            self.impl = MultiHeadCausalSelfAttention(n_embd, n_head)
        else:
            self.impl = SingleHeadCausalSelfAttention(n_embd)

    def forward(self, x):
        return self.impl(x)

class FeedForward(nn.Module):
    def __init__(self,n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.GELU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )
    def forward(self,x):
        return self.net(x)
class Block(nn.Module):
    def __init__(self,n_embd,n_head,use_multi_head_attention=True):
        super().__init__()
        self.sa = CausalSelfAttention(n_embd, n_head, use_multi_head=use_multi_head_attention)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self,x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x
class GPTLanguageModel(nn.Module):
    def __init__(self, use_multi_head_attention=True):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks=nn.Sequential(*[Block(n_embd, n_head, use_multi_head_attention) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.token_embedding_table.weight=self.lm_head.weight
        self.apply(self._init_weights)
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None):
        B,T=idx.size()
        token_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = token_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            loss=None
        else:
            B,T,C=logits.size()
            logits=logits.view(B*T,C)
            targets=targets.view(B*T)
            loss=F.cross_entropy(logits, targets)
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                k = min(top_k, logits.size(-1))
                v, _ = torch.topk(logits, k)
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx
use_multi_head_attention = True  # set False to use single-head attention
model=GPTLanguageModel(use_multi_head_attention=use_multi_head_attention).to(device)
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

1.31456 M parameters


In [17]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [18]:
import math

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

def get_lr(it):
    warmup_iters = 50
    min_lr = 1e-4
    if it < warmup_iters:
        return learning_rate * it / warmup_iters
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return max(min_lr, learning_rate * coeff)

for iter in range(max_iters):
    lr = get_lr(iter)
    optimizer.param_groups[0]['lr'] = lr
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"\nFinal loss: {loss.item():.4f}")

step 0: train loss 8.5997, val loss 8.5994
step 50: train loss 3.4817, val loss 3.4795
step 100: train loss 3.0989, val loss 3.0955
step 150: train loss 2.7789, val loss 2.7857
step 200: train loss 2.6775, val loss 2.6736
step 250: train loss 2.6544, val loss 2.6300
step 300: train loss 2.6042, val loss 2.5894
step 350: train loss 2.5759, val loss 2.5531
step 400: train loss 2.5366, val loss 2.5324
step 450: train loss 2.5285, val loss 2.5135

Final loss: 2.5887


# fine tune 

create python file in colab 

In [19]:
%%writefile my_script.py

print("Hello from Python file")

for i in range(5):
    print(i)

Writing my_script.py


In [20]:
! python my_script.py

Hello from Python file
0
1
2
3
4


In [1]:
%%writefile training.py


import torch
import torch.nn as nn
import torch.nn.functional as F
import mmap 
import random
import pickle 
import argparse 
parser = argparse.ArgumentParser(description='Train a GPT-like model on the OpenWebText dataset.')
parser.add_argument('batch_size', type=int, default=8,required=True, help='Batch size for training')
args = parser.parse_args()
batch_size = args.batch_size
block_size=128 
max_iters=200 
learning_rate=3e-4 
eval_iters=100 
n_embd=384 
n_head=1
n_layer=1 
dropout = 0.2
print(device)

chars=""
with open("/content/vocab.txt","r",encoding="utf-8") as f:
    chars=f.read()
chars=sorted(set(chars))
text=f.read()
string_to_int={ch:i for i,ch in enumerate(chars)}
int_to_string={i:ch for i,ch in enumerate(chars)}
encode=lambda s:[string_to_int[c] for c in s]
decode=lambda l:"".join([int_to_string[i] for i in l])
vocab_size=len(chars)
print(f"vocab_size: {vocab_size}")
print("encode('hello world'):", encode("hello world"))
def get_random_chunk(split):
    path="/content/output_train.txt" if split=="train" else "/content/output_val.txt"
    with open(path,"r",encoding="utf-8") as f:
        data=f.read()
        with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
            file_size = mm.size()
            start_pos=random.randint(0,(file_size)-batch_size*block_size)
            mm.seek(start_pos)
            block=mm.read(block_size*batch_size-1).decode("utf-8", errors="ignore")
            decoded_block=block.decode("utf-8", errors="ignore").replace("r"," ")
            data=torch.tensor(encode(decoded_block), dtype=torch.long)
return data

def get_batch(split):
    data=get_random_chunk(split)
    ix=torch.randint(len(data)-block_size, (batch_size,))
    x=torch.stack([data[i:i+block_size] for i in ix])
    y=torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)
   
   
@ torch.no_grad()
def estimate_loss():
    out={}
    model.eval()
    for split in ['train','val']:
        losses=torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y=get_batch(split)
            _,loss=model(X,Y)
            losses[k]=loss.item()
        out[split]=losses.mean()
    model.train()
    return out

model=GPTLanguageModel(vocab)
print('load the model')
with open("/content/model.pkl","rb") as f:
    model=pickle.load(f)
model=model.to(device)

while True:
    prompt=input("Enter prompt (or 'exit' to quit): ")
    context=torch.tensor(encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
    generated=model.generate(context, max_new_tokens=100, temperature=0.8, top_k=40)
    print(decode(generated[0].tolist()))

Writing training.py


In [4]:

%%writefile training.py


import argparse
import pickle
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
block_size = 128
n_embd = 384
n_head = 1
n_layer = 1
dropout = 0.2
vocab_size = 0
string_to_int = {}
int_to_string = {}


def build_parser():
    parser = argparse.ArgumentParser(
        description="Load a saved GPT-like model and generate text."
    )
    parser.add_argument(
        "--vocab-path",
        type=Path,
        default=Path("vocab.txt"),
        help="Path to the vocabulary file",
    )
    parser.add_argument(
        "--model-path",
        type=Path,
        default=Path("model.pkl"),
        help="Path to the saved model file",
    )
    parser.add_argument(
        "--prompt",
        type=str,
        default=None,
        help="Prompt to generate from. If omitted, interactive mode is used.",
    )
    parser.add_argument(
        "--max-new-tokens",
        type=int,
        default=100,
        help="Number of tokens to generate",
    )
    parser.add_argument(
        "--temperature",
        type=float,
        default=0.8,
        help="Sampling temperature",
    )
    parser.add_argument(
        "--top-k",
        type=int,
        default=40,
        help="Top-k sampling cutoff. Use 0 to disable.",
    )
    parser.add_argument(
        "--device",
        type=str,
        default=None,
        choices=("cpu", "cuda", "mps"),
        help="Override the device selection",
    )
    parser.add_argument("--block-size", type=int, default=128)
    parser.add_argument("--n-embd", type=int, default=384)
    parser.add_argument("--n-head", type=int, default=1)
    parser.add_argument("--n-layer", type=int, default=1)
    parser.add_argument("--dropout", type=float, default=0.2)
    return parser


def configure_runtime(args):
    global block_size, n_embd, n_head, n_layer, dropout, device

    block_size = args.block_size
    n_embd = args.n_embd
    n_head = args.n_head
    n_layer = args.n_layer
    dropout = args.dropout

    if args.device is not None:
        requested = torch.device(args.device)
        if requested.type == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CUDA was requested but is not available.")
        if requested.type == "mps" and not torch.backends.mps.is_available():
            raise RuntimeError("MPS was requested but is not available.")
        device = requested


def load_vocab(vocab_path):
    global vocab_size, string_to_int, int_to_string

    chars = vocab_path.read_text(encoding="utf-8")
    if not chars:
        raise ValueError(f"Vocabulary file is empty: {vocab_path}")

    chars = sorted(set(chars))
    string_to_int = {ch: i for i, ch in enumerate(chars)}
    int_to_string = {i: ch for i, ch in enumerate(chars)}
    vocab_size = len(chars)
    print(f"Using device: {device}")
    print(f"Loaded vocab from: {vocab_path}")
    print(f"vocab_size: {vocab_size}")


def encode(text):
    missing = sorted({ch for ch in text if ch not in string_to_int})
    if missing:
        preview = "".join(missing[:10])
        raise ValueError(
            f"Prompt contains characters that are not in the vocab: {preview!r}"
        )
    return [string_to_int[ch] for ch in text]


def decode(tokens):
    return "".join(int_to_string[token] for token in tokens)


class SingleHeadCausalSelfAttention(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.q_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.k_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.v_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        q = self.q_proj(x).unsqueeze(1)
        k = self.k_proj(x).unsqueeze(1)
        v = self.v_proj(x).unsqueeze(1)
        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=dropout if self.training else 0.0,
            is_causal=True,
        ).squeeze(1)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        if n_embd % n_head != 0:
            raise ValueError("n_embd must be divisible by n_head.")

        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.q_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.k_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.v_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_len, channels = x.size()
        q = self.q_proj(x).view(
            batch_size, seq_len, self.n_head, self.head_dim
        ).transpose(1, 2)
        k = self.k_proj(x).view(
            batch_size, seq_len, self.n_head, self.head_dim
        ).transpose(1, 2)
        v = self.v_proj(x).view(
            batch_size, seq_len, self.n_head, self.head_dim
        ).transpose(1, 2)

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=dropout if self.training else 0.0,
            is_causal=True,
        )
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, channels)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        if n_head > 1:
            self.impl = MultiHeadCausalSelfAttention(n_embd, n_head)
        else:
            self.impl = SingleHeadCausalSelfAttention(n_embd)

    def forward(self, x):
        return self.impl(x)


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.sa = CausalSelfAttention(n_embd, n_head)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.token_embedding_table.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        _, seq_len = idx.size()
        token_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(seq_len, device=device))
        x = token_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            return logits, None

        batch_size, seq_len, channels = logits.size()
        loss = F.cross_entropy(
            logits.view(batch_size * seq_len, channels),
            targets.view(batch_size * seq_len),
        )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                limit = min(top_k, logits.size(-1))
                values, _ = torch.topk(logits, limit)
                logits[logits < values[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx


def load_model(model_path):
    suffix = model_path.suffix.lower()
    if suffix in {".pt", ".pth", ".bin"}:
        payload = torch.load(model_path, map_location=device)
        if isinstance(payload, nn.Module):
            model = payload
        else:
            model = GPTLanguageModel()
            if isinstance(payload, dict) and "state_dict" in payload:
                payload = payload["state_dict"]
            model.load_state_dict(payload)
    else:
        with model_path.open("rb") as handle:
            model = pickle.load(handle)

    if not isinstance(model, nn.Module):
        raise TypeError(f"Loaded object is not a torch model: {type(model)!r}")

    model = model.to(device)
    model.eval()
    print(f"Loaded model from: {model_path}")
    return model


def generate_text(model, prompt, max_new_tokens, temperature, top_k):
    encoded_prompt = encode(prompt)
    context = torch.tensor(encoded_prompt, dtype=torch.long, device=device).unsqueeze(0)
    generated = model.generate(
        context,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
    )
    return decode(generated[0].tolist())


def main():
    args = build_parser().parse_args()
    configure_runtime(args)

    if not args.vocab_path.is_file():
        raise FileNotFoundError(f"Vocabulary file not found: {args.vocab_path}")
    if not args.model_path.is_file():
        raise FileNotFoundError(f"Model file not found: {args.model_path}")

    load_vocab(args.vocab_path)
    model = load_model(args.model_path)
    top_k = None if args.top_k <= 0 else args.top_k

    if args.prompt is not None:
        print(
            generate_text(
                model,
                args.prompt,
                max_new_tokens=args.max_new_tokens,
                temperature=args.temperature,
                top_k=top_k,
            )
        )
        return

    while True:
        prompt = input("Enter prompt (or 'exit' to quit): ").strip()
        if prompt.lower() == "exit":
            break
        if not prompt:
            continue
        print(
            generate_text(
                model,
                prompt,
                max_new_tokens=args.max_new_tokens,
                temperature=args.temperature,
                top_k=top_k,
            )
        )


if __name__ == "__main__":
    main()


Overwriting training.py


In [5]:
!python training.py --batch_size 16

: 